In [3]:
!pip install krakenex

import krakenex

# Create an instance of the Kraken API
api = krakenex.API()

# Fetch all available asset pairs
response = api.query_public('AssetPairs')

if 'result' in response:
    asset_pairs = list(response['result'].keys())
    print("Available pairs on Kraken:")
    for pair in asset_pairs:
        print(pair)
else:
    print(f"Error fetching asset pairs: {response.get('error', 'Unknown error')}")

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py install for krakenex: started
  Running setup.py install for krakenex: finished with status 'done'


  DEPRECATION: krakenex is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559

[notice] A new release of pip available: 22.3.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Available pairs on Kraken:
1INCHEUR
1INCHUSD
AAVEETH
AAVEEUR
AAVEGBP
AAVEUSD
AAVEXBT
ACAEUR
ACAUSD
ACHEUR
ACHUSD
ADAAUD
ADAETH
ADAEUR
ADAGBP
ADAUSD
ADAUSDC
ADAUSDT
ADAXBT
ADXEUR
ADXUSD
AEVOEUR
AEVOUSD
AGLDEUR
AGLDUSD
AIREUR
AIRUSD
AKTEUR
AKTUSD
ALCXEUR
ALCXUSD
ALGOETH
ALGOEUR
ALGOGBP
ALGOUSD
ALGOUSDC
ALGOUSDT
ALGOXBT
ALICEEUR
ALICEUSD
ALPHAEUR
ALPHAUSD
ALTEUR
ALTUSD
ANKREUR
ANKRUSD
ANKRXBT
APEEUR
APEUSD
APEUSDC
APEUSDT
API3EUR
API3USD
APTEUR
APTUSD
APUEUR
APUUSD
ARBEUR
ARBUSD
ARKMEUR
ARKMUSD
ARPAEUR
ARPAUSD
ASTREUR
ASTRUSD
ATHEUR
ATHUSD
ATLASEUR
ATLASUSD
ATOMETH
ATOMEUR
ATOMGBP
ATOMUSD
ATOMUSDC
ATOMUSDT
ATOMXBT
AUCTIONEUR
AUCTIONUSD
AUDIOEUR
AUDIOUSD
AUDJPY
AUDUSD
AVAXEUR
AVAXUSD
AVAXUSDC
AVAXUSDT
AXSEUR
AXSUSD
BADGEREUR
BADGERUSD
BALEUR
BALUSD
BANDEUR
BANDUSD
BATEUR
BATUSD
BCHAUD
BCHETH
BCHEUR
BCHGBP
BCHJPY
BCHUSD
BCHUSDC
BCHUSDT
BCHXBT
BEAMEUR
BEAMUSD
BICOEUR
BICOUSD
BIGTIMEEUR
BIGTIMEUSD
BITEUR
BITUSD
BLUREUR
BLURUSD
BLZEUR
BLZUSD
BNCEUR
BNCUSD
BNTEUR
BNTUSD
BOBAEUR
BOBAUSD
BODENEUR

In [5]:
import krakenex
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

class KrakenData:
    def __init__(self):
        self.api = krakenex.API()
    
    def fetch_data(self, pair):
        try:
            response = self.api.query_public('OHLC', {'pair': pair, 'interval': 15})
            if response['error']:
                raise Exception(f"API Error: {response['error']}")
            data = response['result'][list(response['result'].keys())[0]]
            df = pd.DataFrame(data, columns=['time', 'open', 'high', 'low', 'close', 'vwap', 'volume', 'count'])
            df['time'] = pd.to_datetime(df['time'], unit='s')
            df['close'] = df['close'].astype(float)
            return df[['time', 'close']]
        except Exception as e:
            print(f"Error fetching data: {e}")
            return None

class IndicatorCalculator:
    @staticmethod
    def calculate_bollinger_bands(df, window=20):
        try:
            df['mean'] = df['close'].rolling(window=window).mean()
            df['std'] = df['close'].rolling(window=window).std()
            df['upper_band'] = df['mean'] + 2 * df['std']
            df['lower_band'] = df['mean'] - 2 * df['std']
            return df
        except Exception as e:
            print(f"Error calculating Bollinger Bands: {e}")
            return None
    
    @staticmethod
    def generate_signals(df):
        try:
            df['signal'] = None
            df.loc[df['close'] <= df['lower_band'], 'signal'] = 'Buy'
            df.loc[df['close'] >= df['upper_band'], 'signal'] = 'Sell'
            return df
        except Exception as e:
            print(f"Error generating signals: {e}")
            return None

class Visualizer:
    @staticmethod
    def plot_data(df, pair):
        try:
            plt.figure(figsize=(12, 6))
            plt.plot(df['time'], df['close'], label='Close Price', color='blue')
            plt.plot(df['time'], df['upper_band'], label='Upper Band', color='red', linestyle='--')
            plt.plot(df['time'], df['lower_band'], label='Lower Band', color='green', linestyle='--')
            buy_signals = df[df['signal'] == 'Buy']
            sell_signals = df[df['signal'] == 'Sell']
            plt.scatter(buy_signals['time'], buy_signals['close'], label='Buy Signal', marker='^', color='green')
            plt.scatter(sell_signals['time'], sell_signals['close'], label='Sell Signal', marker='v', color='red')
            plt.title(f"Bollinger Bands and Signals for {pair}")
            plt.xlabel('Time')
            plt.ylabel('Price')
            plt.legend()
            plt.grid()
            plt.show()
        except Exception as e:
            print(f"Error plotting data: {e}")

if __name__ == "__main__":
    # Menú interactivo o ingreso manual del par de monedas
    pairs = ['XXBTZEUR', 'XETHZEUR', 'XLTCZEUR']
    print("Available pairs:")
    for i, p in enumerate(pairs):
        print(f"{i + 1}. {p}")
    print("0. Enter a custom pair")
    choice = int(input("Select an option (0-3): "))
    if choice == 0:
        pair = input("Enter the currency pair (e.g., XXBTZEUR): ")
    else:
        pair = pairs[choice - 1]

    # Inicializar clases
    kraken_data = KrakenData()
    calculator = IndicatorCalculator()

    # Descargar datos
    df = kraken_data.fetch_data(pair)
    if df is not None:
        # Calcular indicadores
        df = calculator.calculate_bollinger_bands(df)
        df = calculator.generate_signals(df)
        # Graficar resultados
        Visualizer.plot_data(df, pair)


Available pairs:
1. XXBTZEUR
2. XETHZEUR
3. XLTCZEUR
0. Enter a custom pair


ValueError: invalid literal for int() with base 10: 'ACHEUR'

In [4]:
import krakenex
import pandas as pd
import numpy as np
import plotly.graph_objects as go

class KrakenData:
    def __init__(self):
        self.api = krakenex.API()
    
    def fetch_data(self, pair):
        try:
            response = self.api.query_public('OHLC', {'pair': pair, 'interval': 15})
            if response['error']:
                raise Exception(f"API Error: {response['error']}")
            data = response['result'][list(response['result'].keys())[0]]
            df = pd.DataFrame(data, columns=['time', 'open', 'high', 'low', 'close', 'vwap', 'volume', 'count'])
            df['time'] = pd.to_datetime(df['time'], unit='s')
            df['close'] = df['close'].astype(float)
            return df[['time', 'close']]
        except Exception as e:
            print(f"Error fetching data: {e}")
            return None

class IndicatorCalculator:
    @staticmethod
    def calculate_bollinger_bands(df, window=20):
        try:
            df['mean'] = df['close'].rolling(window=window).mean()
            df['std'] = df['close'].rolling(window=window).std()
            df['upper_band'] = df['mean'] + 2 * df['std']
            df['lower_band'] = df['mean'] - 2 * df['std']
            return df
        except Exception as e:
            print(f"Error calculating Bollinger Bands: {e}")
            return None
    
    @staticmethod
    def generate_signals(df):
        try:
            df['signal'] = None
            df.loc[df['close'] <= df['lower_band'], 'signal'] = 'Buy'
            df.loc[df['close'] >= df['upper_band'], 'signal'] = 'Sell'
            return df
        except Exception as e:
            print(f"Error generating signals: {e}")
            return None

class Visualizer:
    @staticmethod
    def plot_data(df, pair):
        try:
            fig = go.Figure()

            # Add close price trace
            fig.add_trace(go.Scatter(x=df['time'], y=df['close'], mode='lines', name='Close Price', line=dict(color='blue')))
            
            # Add Bollinger Bands traces
            fig.add_trace(go.Scatter(x=df['time'], y=df['upper_band'], mode='lines', name='Upper Band', line=dict(color='red', dash='dash')))
            fig.add_trace(go.Scatter(x=df['time'], y=df['lower_band'], mode='lines', name='Lower Band', line=dict(color='green', dash='dash')))
            
            # Add buy and sell signals
            buy_signals = df[df['signal'] == 'Buy']
            sell_signals = df[df['signal'] == 'Sell']
            fig.add_trace(go.Scatter(x=buy_signals['time'], y=buy_signals['close'], mode='markers', name='Buy Signal', marker=dict(color='green', symbol='triangle-up')))
            fig.add_trace(go.Scatter(x=sell_signals['time'], y=sell_signals['close'], mode='markers', name='Sell Signal', marker=dict(color='red', symbol='triangle-down')))
            
            fig.update_layout(title=f"Bollinger Bands and Signals for {pair}", xaxis_title='Time', yaxis_title='Price', legend_title='Legend')
            fig.show()
        except Exception as e:
            print(f"Error plotting data: {e}")

if __name__ == "__main__":
    # Menú interactivo o ingreso manual del par de monedas
    pairs = ['XXBTZEUR', 'XETHZEUR', 'XLTCZEUR']
    print("Available pairs:")
    for i, p in enumerate(pairs):
        print(f"{i + 1}. {p}")
    print("0. Enter a custom pair")
    choice = int(input("Select an option (0-3): "))
    if choice == 0:
        pair = input("Enter the currency pair (e.g., XXBTZEUR): ")
    else:
        pair = pairs[choice - 1]

    # Inicializar clases
    kraken_data = KrakenData()
    calculator = IndicatorCalculator()

    # Descargar datos
    df = kraken_data.fetch_data(pair)
    if df is not None:
        # Calcular indicadores
        df = calculator.calculate_bollinger_bands(df)
        df = calculator.generate_signals(df)
        # Graficar resultados
        Visualizer.plot_data(df, pair)


Available pairs:
1. XXBTZEUR
2. XETHZEUR
3. XLTCZEUR
0. Enter a custom pair
Error plotting data: Mime type rendering requires nbformat>=4.2.0 but it is not installed
